# Rabi Oscillations with Pulse Streamer
Clean and streamlined interface for performing Rabi oscillation measurements

## 1. Setup and Configuration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pulsestreamer import PulseStreamer, Sequence

# Initialize Pulse Streamer connection
PULSE_STREAMER_IP = '169.254.8.2'  # Update with your device IP
ps = PulseStreamer(PULSE_STREAMER_IP)

In [ ]:
# Hardware Channel Configuration
CHANNELS = {
    'rabi_pulse': 1,
    'trigger': 5
}

# Timing Parameters (in microseconds)
TRIGGER_DURATION = 0.05  # 50 ns
WAIT_TIME = 50.0         # Wait time after pulse

## 2. Rabi Sweep Configuration

In [ ]:
# Rabi Pulse Duration Sweep
rabi_start = 2.0    # Start duration (µs)
rabi_end = 22.0     # End duration (µs)
rabi_steps = 50     # Number of steps

rabi_durations = np.linspace(rabi_start, rabi_end, rabi_steps)
print(f"Rabi sweep: {rabi_start} to {rabi_end} µs in {rabi_steps} steps")

## 3. Sequence Generation

In [ ]:
def create_rabi_sequence(rabi_duration_us):
    """
    Create a single Rabi pulse sequence.
    
    Args:
        rabi_duration_us: Duration of Rabi pulse in microseconds
    
    Returns:
        List of pulse sequence steps
    """
    sequence = []
    
    # Calculate pulse after trigger
    pulse_after_trigger = max(0, rabi_duration_us - TRIGGER_DURATION)
    
    # 1. Trigger + Rabi Pulse Start
    sequence.append({
        'duration_us': TRIGGER_DURATION,
        'channels_on': ['rabi_pulse', 'trigger']
    })
    
    # 2. Continue Rabi Pulse
    if pulse_after_trigger > 0:
        sequence.append({
            'duration_us': pulse_after_trigger,
            'channels_on': ['rabi_pulse']
        })
    
    # 3. Wait for measurement
    sequence.append({
        'duration_us': WAIT_TIME,
        'channels_on': []
    })
    
    return sequence

# Test sequence generation
test_seq = create_rabi_sequence(10.0)
print(f"Test sequence created with {len(test_seq)} steps")

In [ ]:
def create_full_rabi_sweep():
    """
    Create complete Rabi sweep sequence.
    
    Returns:
        Pulse Streamer sequence object
    """
    seq = Sequence()
    
    for duration in rabi_durations:
        steps = create_rabi_sequence(duration)
        
        for step in steps:
            # Convert to nanoseconds
            duration_ns = int(step['duration_us'] * 1000)
            
            # Create channel state
            state = 0
            for ch_name in step.get('channels_on', []):
                state |= (1 << CHANNELS[ch_name])
            
            # Add to sequence
            seq.setDigital(state, duration_ns)
    
    return seq

# Create the full sequence
rabi_sequence = create_full_rabi_sweep()
print("Full Rabi sweep sequence created")

## 4. Visualization

In [ ]:
def plot_rabi_sequence(num_steps=3):
    """
    Plot first few steps of Rabi sequence.
    
    Args:
        num_steps: Number of sweep steps to plot
    """
    fig, ax = plt.subplots(figsize=(15, 6))
    
    # Generate data for plotting
    time_data = {'rabi_pulse': [], 'trigger': []}
    current_time = 0
    
    for i, duration in enumerate(rabi_durations[:num_steps]):
        steps = create_rabi_sequence(duration)
        
        for step in steps:
            step_duration = step['duration_us']
            channels_on = step.get('channels_on', [])
            
            for ch_name in CHANNELS.keys():
                level = 1 if ch_name in channels_on else 0
                time_data[ch_name].extend([current_time, current_time + step_duration])
            
            current_time += step_duration
    
    # Plot channels
    for i, (ch_name, ch_num) in enumerate(CHANNELS.items()):
        times = time_data[ch_name]
        values = [i] * len(times)
        ax.plot(times, values, label=f'CH{ch_num}: {ch_name}', linewidth=2)
    
    ax.set_xlabel('Time (µs)', fontsize=12)
    ax.set_ylabel('Channel', fontsize=12)
    ax.set_title(f'Rabi Sequence (First {num_steps} Steps)', fontsize=14)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

# Plot first 3 steps
plot_rabi_sequence(3)

## 5. Run Measurement

In [ ]:
# Upload sequence to Pulse Streamer
ps.stream(rabi_sequence, n_runs=1)
print("Sequence uploaded to Pulse Streamer")
print(f"Total sweep points: {len(rabi_durations)}")
print(f"Rabi duration range: {rabi_start} - {rabi_end} µs")

In [ ]:
# Start the measurement
ps.startNow()
print("Rabi oscillation measurement started!")

## 6. Data Analysis (Placeholder)

In [ ]:
# TODO: Add your data acquisition and analysis code here
# Example:
# counts = acquire_counts()  # Your DAQ function
# plt.plot(rabi_durations, counts)
# plt.xlabel('Rabi Pulse Duration (µs)')
# plt.ylabel('Counts')
# plt.title('Rabi Oscillations')
# plt.show()

print("Add your data analysis code here")